# Coevolution with Bayesian Updates

In [80]:
import numpy as np
# CONSTANTS: defining constants for evolutionary algorithm parameters

INITIAL_CODE_POPULATION_SIZE = 10
INITIAL_TEST_POPULATION_SIZE = 20

C_0 = 0.6 # Prior probability of a code being correct
T_0 = 0.6 # Prior probability of a test being correct

# Simulation constants
PROBABILITY_OF_CODE_TEST_PASSING = 0.7

In [81]:
import numpy as np

def get_probabilities(log_odds: np.ndarray) -> np.ndarray:
    odds = np.exp(log_odds)
    probabilities = odds / (1 + odds)
    return probabilities

log_C_0 = np.log(C_0/(1-C_0))
log_T_0 = np.log(T_0/(1-T_0))


log_C: np.ndarray = np.array([log_C_0 for _ in range(INITIAL_CODE_POPULATION_SIZE)])
log_T: np.ndarray = np.array([log_T_0 for _ in range(INITIAL_TEST_POPULATION_SIZE)])

print("Initial Code Population Log-Odds:", log_C)
print("Initial Test Population Log-Odds:", log_T)

# Convert log-odds to probabilities
C = get_probabilities(log_C)
T = get_probabilities(log_T)    

print("Initial Code Population Probabilities:", C)
print("Initial Test Population Probabilities:", T)

Initial Code Population Log-Odds: [0.40546511 0.40546511 0.40546511 0.40546511 0.40546511 0.40546511
 0.40546511 0.40546511 0.40546511 0.40546511]
Initial Test Population Log-Odds: [0.40546511 0.40546511 0.40546511 0.40546511 0.40546511 0.40546511
 0.40546511 0.40546511 0.40546511 0.40546511 0.40546511 0.40546511
 0.40546511 0.40546511 0.40546511 0.40546511 0.40546511 0.40546511
 0.40546511 0.40546511]
Initial Code Population Probabilities: [0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6]
Initial Test Population Probabilities: [0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6
 0.6 0.6]


# Initial Code-Test Evaluation

In [82]:
def _evaluate_code_test_pair(code_idx: int, test_idx: int) -> int:
    # Simulation of code-test evaluation
    # Randomly return True (pass) or False (fail)
    return int(np.random.rand() < PROBABILITY_OF_CODE_TEST_PASSING)

def evaluate_population(C: np.ndarray, T: np.ndarray) -> np.ndarray:
    matrix = np.zeros((C.shape[0], T.shape[0]), dtype=np.int8)
    for i in range(C.shape[0]):
        for j in range(T.shape[0]):
            matrix[i, j] = _evaluate_code_test_pair(i, j)
    return matrix

matrix = evaluate_population(C, T)
matrix  # Convert boolean to int for better visualization

array([[1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0],
       [0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
       [1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1],
       [1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1],
       [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1],
       [1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1]],
      dtype=int8)

# Posterior Probability Update

In [83]:
# def _get_posterior_update(X: float, Y: float, observation: bool) -> float:
#     """
#     Computes the updated belief for a code or test.
#     Args:
#         X: Previous belief that we will be updating
#         Y: Previous belief that determines the update (e.g., if updating code belief, Y is test belief)
#         observation: Result of the test case (True for pass, False for fail)
#     """

#     if observation:
#         k = (1 - Y) / Y
#     else:
#         k = Y / (1 - Y)

#     ratio = (k) * ((1 - X) / X)
#     return 1 / (1 + ratio)


# def get_posterior_update_for_code(code_prob: float, test_prob:float, observation: bool) -> float:
#     return _get_posterior_update(code_prob, test_prob, observation)

# def get_posterior_update_for_test(code_prob:float, test_prob: float, observation: bool) -> float:
#     return _get_posterior_update(test_prob, code_prob, observation)


# def update_all_probabilities(C_prev: np.ndarray, T_prev: np.ndarray, matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
#     """
#     Demonstrates the Bayesian update for all codes and tests based on the evaluation matrix.
#     Args:
#         C: Array of probabilities that each code is correct
#         T: Array of probabilities that each test is correct
#         matrix: Evaluation matrix where each entry is 1 (pass) or 0 (fail)
#     """

#     C_new = np.zeros_like(C_prev)
#     T_new = np.zeros_like(T_prev)


#     print("=== Posterior Update ===")
#     print("--- Updating Code Probabilities ---")
#     for i in range(C_prev.shape[0]):
#         print("Previous Code Probability:", C_prev[i])

#         code_prob: float = C_prev[i]
#         for j in range(T_prev.shape[0]):
#             test_prob: float = T_prev[j]
#             observation = bool(matrix[i, j])

#             code_prob = get_posterior_update_for_code(code_prob, test_prob, observation)
#             print(f"  Test {j} (Prob: {test_prob}, Observation: {observation}) -> Updated Code Prob: {code_prob}")
        
#         C_new[i] = code_prob
#         print("Updated Code Probability:", C_new[i])

#     print("--- Updating Test Probabilities ---")
#     for j in range(T_prev.shape[0]):
#         print("Previous Test Probability:", T_prev[j])

#         test_prob = T_prev[j]
#         for i in range(C_prev.shape[0]):
#             code_prob = C_prev[i]
#             observation = bool(matrix[i, j])

#             test_prob = get_posterior_update_for_test(code_prob, test_prob, observation)
#             print(f"  Code {i} (Prob: {code_prob}, Observation: {observation}) -> Updated Test Prob: {test_prob}")
        
#         T_new[j] = test_prob
#         print("Updated Test Probability:", T_new[j])




#     return C_new, T_new
    
# for i in range(1):
#     print(f"=== Iteration {i+1} ===")
#     C, T = update_all_probabilities(C, T, matrix)
#     print("Updated Code Population Probabilities:", C)
#     print("Updated Test Population Probabilities:", T)

# Log Updates

In [84]:
def _get_posterior_log_odds(prior_log_odds: float, correctness_log_odds: np.ndarray, observations: np.ndarray) -> float:
    """
    Calculates the posterior log-odds given prior log-odds, other entity's log-odds corresponding to each observation, and observations.
    Args:
        prior_log_odds: Prior log-odds of the entity being correct
        correctness_log_odds: Array of log-odds for the other entity (e.g., if updating code, this is test log-odds)
        observations: Array of observations (1 for pass, 0 for fail)
    Returns:
        Updated log-odds after considering all observations
    """

    if len(correctness_log_odds) != len(observations):
        raise ValueError("Length of correctness_log_odds and observations must be the same")


    # If an observation is 1 (pass/true), its weight is the correctness log-odds.
    # If an observation is 0 (fail/false), it provides evidence against the hypothesis,
    # so its weight is the negative of the correctness log-odds.
    evidence_weights = np.where(observations == 1, correctness_log_odds, -correctness_log_odds)

    # Sum the weights of all individual observations.
    sum_of_weights: float = np.sum(evidence_weights)

    # The posterior log-odds is the prior plus the total weight of new evidence.
    posterior_log_odds: float = prior_log_odds + sum_of_weights

    return posterior_log_odds

def get_posterior_log_odds_for_code(prior_log_odds: float, test_log_odds: np.ndarray, observations: np.ndarray) -> float:
    return _get_posterior_log_odds(prior_log_odds, test_log_odds, observations)

def get_posterior_log_odds_for_test(prior_log_odds: float, code_log_odds: np.ndarray, observations: np.ndarray) -> float:
    return _get_posterior_log_odds(prior_log_odds, code_log_odds, observations)



def update_all_log_odds(log_C_prev: np.ndarray, log_T_prev: np.ndarray, matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Demonstrates the Bayesian update for all codes and tests based on the evaluation matrix using log-odds.
    Args:
        log_C_prev: Array of log-odds that each code is correct
        log_T_prev: Array of log-odds that each test is correct
        matrix: Evaluation matrix where each entry is 1 (pass) or 0 (fail)
    """

    log_C_new = np.zeros_like(log_C_prev)
    log_T_new = np.zeros_like(log_T_prev)


    # print("=== Posterior Log-Odds Update ===")
    # print("--- Updating Code Log-Odds ---")
    for i in range(log_C_prev.shape[0]):

        code_log_odds: float = log_C_prev[i]
        test_log_odds: np.ndarray = log_T_prev
        observations: np.ndarray = matrix[i, :]

        code_log_odds = get_posterior_log_odds_for_code(code_log_odds, test_log_odds, observations)
        
        log_C_new[i] = code_log_odds

    # print("Updated Code Log-Odds:", log_C_new)

    # print("--- Updating Test Log-Odds ---")
    for j in range(log_T_prev.shape[0]):
        test_log_odds: float = log_T_prev[j]
        code_log_odds: np.ndarray = log_C_prev # TODO: We can potentially use the updated code log-odds here
        observations: np.ndarray = matrix[:, j]

        test_log_odds = get_posterior_log_odds_for_test(test_log_odds, code_log_odds, observations)
        
        log_T_new[j] = test_log_odds



    # print("Updated Test Log-Odds:", log_T_new)
    return log_C_new, log_T_new


for i in range(20):
    print(f"=== Iteration {i+1} ===")
    matrix = evaluate_population(C, T)
    log_C, log_T = update_all_log_odds(log_C, log_T, matrix)
    C = get_probabilities(log_C)
    T = get_probabilities(log_T)
    print("Updated Code Population Probabilities:", C)
    print("Updated Test Population Probabilities:", T)

=== Iteration 1 ===
Updated Code Population Probabilities: [0.97464719 0.88363636 0.98857111 0.99488804 0.98857111 0.88363636
 0.88363636 0.97464719 0.99488804 0.98857111]
Updated Test Population Probabilities: [0.88363636 0.77142857 0.94470842 0.97464719 0.97464719 0.94470842
 0.77142857 0.97464719 0.94470842 0.77142857 0.94470842 0.88363636
 0.77142857 0.6        0.94470842 0.6        0.88363636 0.77142857
 0.88363636 0.77142857]
=== Iteration 2 ===
Updated Code Population Probabilities: [0.99999999 1.         0.02535281 1.         1.         0.99999999
 1.         1.         1.         0.99999969]
Updated Test Population Probabilities: [1.00000000e+00 9.99999988e-01 9.99999998e-01 1.00000000e+00
 9.99549110e-01 1.00000000e+00 8.83636364e-01 1.00000000e+00
 1.00000000e+00 1.00000000e+00 9.99999864e-01 1.35656572e-07
 1.00000000e+00 9.99992177e-01 1.01393013e-03 1.00000000e+00
 1.00000000e+00 9.99998455e-01 9.99799554e-01 9.99999695e-01]
=== Iteration 3 ===
Updated Code Population Pro

/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/ipykernel_43119/1871232409.py:4: RuntimeWarning: overflow encountered in exp
  odds = np.exp(log_odds)
/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/ipykernel_43119/1871232409.py:5: RuntimeWarning: invalid value encountered in divide
  probabilities = odds / (1 + odds)
